<cell_type>markdown</cell_type># 负载均衡教程 (Load Balancing Tutorial)

> **前置知识**: Python 异步编程、HTTP 基础、网络基础
>
> **学习目标**: 掌握负载均衡的原理和实现

---

## 什么是负载均衡？

```
┌─────────────────────────────────────────────────────────────┐
│                    负载均衡架构                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│                    ┌─────────────────────┐                  │
│                    │     负载均衡器       │                  │
│                    │ (Nginx/HAProxy/K8s) │                  │
│                    └──────────┬──────────┘                  │
│                               │                             │
│            ┌──────────────────┼──────────────────┐          │
│            │                  │                  │          │
│            ▼                  ▼                  ▼          │
│     ┌───────────┐      ┌───────────┐      ┌───────────┐    │
│     │ 服务实例1  │      │ 服务实例2  │      │ 服务实例3  │    │
│     │ (FastAPI) │      │ (FastAPI) │      │ (FastAPI) │    │
│     └───────────┘      └───────────┘      └───────────┘    │
│                                                             │
│  核心功能:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  1. 流量分发: 将请求分配到多个服务器                │   │
│  │  2. 健康检查: 自动检测并剔除故障节点                │   │
│  │  3. 高可用: 单点故障不影响整体服务                  │   │
│  │  4. 可扩展: 动态增减服务器数量                      │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

## 本教程内容

1. **服务器节点** - Server 类定义
2. **负载均衡策略** - 轮询、加权、最少连接、IP哈希
3. **限流器** - 令牌桶算法
4. **熔断器** - 故障隔离
5. **完整负载均衡器** - 生产级实现

In [ ]:
# ============================================================
# 环境准备
# ============================================================
import asyncio
import time
import random
import hashlib
import numpy as np
from typing import List, Dict, Any, Optional
from dataclasses import dataclass, field
from enum import Enum
from collections import deque
from abc import ABC, abstractmethod

# 设置随机种子
np.random.seed(42)
random.seed(42)

print("=" * 60)
print("环境准备完成")
print("=" * 60)

# 检查 httpx
try:
    import httpx
    HTTPX_AVAILABLE = True
    print(f"\n✓ httpx 已安装")
except ImportError:
    HTTPX_AVAILABLE = False
    print(f"\n✗ httpx 未安装")
    print("  安装命令: pip install httpx")

print(f"\n注意: 本教程的核心组件 (策略、限流、熔断) 不依赖 httpx")
print(f"      可以独立学习和使用")

<cell_type>markdown</cell_type>## 1. 服务器节点

**核心概念**: Server 类表示后端服务器节点，包含健康状态、连接数、响应时间等信息

```
┌─────────────────────────────────────────────────────────────┐
│                    服务器节点属性                            │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  Server:                                                    │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  url: "http://server1:8000"   # 服务器地址          │   │
│  │  weight: 3                    # 权重 (用于加权策略) │   │
│  │  healthy: True                # 健康状态            │   │
│  │  connections: 0               # 当前连接数          │   │
│  │  response_times: [10, 12, 8]  # 最近响应时间        │   │
│  │  success_count: 100           # 成功次数            │   │
│  │  failure_count: 2             # 失败次数            │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  计算属性:                                                  │
│  - avg_response_time: 平均响应时间                         │
│  - success_rate: 成功率 = success / (success + failure)    │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 服务器节点定义 (自包含实现)
# ============================================================
print("=" * 60)
print("服务器节点")
print("=" * 60)

@dataclass
class Server:
    """
    服务器节点
    
    表示后端服务器，包含健康状态、连接数、响应时间等信息
    """
    url: str
    weight: int = 1
    healthy: bool = True
    connections: int = 0
    response_times: deque = field(default_factory=lambda: deque(maxlen=100))
    success_count: int = 0
    failure_count: int = 0
    last_check: float = 0.0
    
    @property
    def avg_response_time(self) -> float:
        """平均响应时间"""
        if not self.response_times:
            return 0.0
        return sum(self.response_times) / len(self.response_times)
    
    @property
    def success_rate(self) -> float:
        """成功率"""
        total = self.success_count + self.failure_count
        if total == 0:
            return 1.0
        return self.success_count / total
    
    def record_response(self, latency_ms: float, success: bool = True):
        """记录响应"""
        self.response_times.append(latency_ms)
        if success:
            self.success_count += 1
        else:
            self.failure_count += 1


# 创建服务器节点示例
servers = [
    Server(url="http://server1:8000", weight=3),
    Server(url="http://server2:8000", weight=2),
    Server(url="http://server3:8000", weight=1),
]

print("\n服务器节点:")
for s in servers:
    print(f"  {s.url}: 权重={s.weight}, 健康={s.healthy}")

In [ ]:
# ============================================================
# 测试服务器响应记录
# ============================================================
print("=" * 60)
print("服务器响应记录测试")
print("=" * 60)

# 模拟记录响应
server = servers[0]

# 记录一些响应
for i in range(20):
    latency = np.random.exponential(10)  # 模拟延迟 (指数分布)
    success = np.random.random() > 0.1   # 90% 成功率
    server.record_response(latency, success)

print(f"\n服务器统计 ({server.url}):")
print(f"  平均响应时间: {server.avg_response_time:.2f}ms")
print(f"  成功率: {server.success_rate:.1%}")
print(f"  成功次数: {server.success_count}")
print(f"  失败次数: {server.failure_count}")

<cell_type>markdown</cell_type>## 2. 负载均衡策略

**核心概念**: 不同的策略适用于不同的场景

```
┌─────────────────────────────────────────────────────────────┐
│                    负载均衡策略对比                          │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  策略              原理                  适用场景           │
│  ────              ────                  ────────           │
│  轮询              依次选择              服务器性能相近     │
│  加权轮询          按权重分配            服务器性能不同     │
│  最少连接          选择连接数最少        长连接场景         │
│  IP 哈希           相同 IP 路由相同      有状态服务         │
│  响应时间          选择响应最快          延迟敏感场景       │
│  随机              随机选择              无状态服务         │
│                                                             │
│  轮询示意:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  请求 1 → Server 1                                  │   │
│  │  请求 2 → Server 2                                  │   │
│  │  请求 3 → Server 3                                  │   │
│  │  请求 4 → Server 1  (循环)                          │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### 2.1 轮询策略 (Round Robin)

In [ ]:
# ============================================================
# 负载均衡策略基类和实现 (自包含实现)
# ============================================================
print("=" * 60)
print("负载均衡策略")
print("=" * 60)

class LoadBalanceStrategy(ABC):
    """负载均衡策略基类"""
    
    @abstractmethod
    def select(self, servers: List[Server], client_ip: str = None) -> Server:
        """选择一个服务器"""
        pass
    
    def _get_healthy_servers(self, servers: List[Server]) -> List[Server]:
        """获取健康的服务器"""
        return [s for s in servers if s.healthy]


class RoundRobinStrategy(LoadBalanceStrategy):
    """轮询策略: 按顺序依次选择服务器"""
    
    def __init__(self):
        self.current_index = 0
    
    def select(self, servers: List[Server], client_ip: str = None) -> Server:
        healthy = self._get_healthy_servers(servers)
        if not healthy:
            raise RuntimeError("No healthy servers available")
        
        server = healthy[self.current_index % len(healthy)]
        self.current_index += 1
        return server


# 测试轮询策略
rr_strategy = RoundRobinStrategy()

rr_servers = [
    Server(url="http://server1:8000"),
    Server(url="http://server2:8000"),
    Server(url="http://server3:8000"),
]

print("\n轮询策略选择顺序:")
for i in range(9):
    selected = rr_strategy.select(rr_servers)
    print(f"  请求 {i+1}: {selected.url}")

<cell_type>markdown</cell_type>### 2.2 加权轮询策略 (Weighted Round Robin)

**核心概念**: 根据权重分配请求，权重高的服务器处理更多请求

```
┌─────────────────────────────────────────────────────────────┐
│                    加权轮询示意                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  权重配置: Server1=3, Server2=2, Server3=1                  │
│  总权重: 6                                                  │
│                                                             │
│  请求分配:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  请求 1-3 → Server 1 (权重 3, 占 50%)              │   │
│  │  请求 4-5 → Server 2 (权重 2, 占 33%)              │   │
│  │  请求 6   → Server 3 (权重 1, 占 17%)              │   │
│  │  请求 7-9 → Server 1 (循环)                        │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 加权轮询策略
# ============================================================
print("=" * 60)
print("加权轮询策略")
print("=" * 60)

class WeightedRoundRobinStrategy(LoadBalanceStrategy):
    """加权轮询策略: 根据权重分配请求"""
    
    def __init__(self):
        self.current_index = 0
    
    def select(self, servers: List[Server], client_ip: str = None) -> Server:
        healthy = self._get_healthy_servers(servers)
        if not healthy:
            raise RuntimeError("No healthy servers available")
        
        # 计算总权重
        total_weight = sum(s.weight for s in healthy)
        point = self.current_index % total_weight
        self.current_index += 1
        
        # 根据权重选择
        current = 0
        for server in healthy:
            current += server.weight
            if point < current:
                return server
        return healthy[-1]


# 测试加权轮询
wrr_strategy = WeightedRoundRobinStrategy()

weighted_servers = [
    Server(url="http://server1:8000", weight=3),  # 50%
    Server(url="http://server2:8000", weight=2),  # 33%
    Server(url="http://server3:8000", weight=1),  # 17%
]

# 统计分布
counts = {s.url: 0 for s in weighted_servers}
for _ in range(600):
    selected = wrr_strategy.select(weighted_servers)
    counts[selected.url] += 1

print("\n加权轮询分布 (权重 3:2:1):")
for url, count in counts.items():
    print(f"  {url}: {count} 次 ({count/600:.1%})")

<cell_type>markdown</cell_type>### 2.3 最少连接策略 (Least Connections)

**核心概念**: 选择当前连接数最少的服务器，适用于长连接场景

```
┌─────────────────────────────────────────────────────────────┐
│                    最少连接策略                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  服务器状态:                                                │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  Server 1: 10 连接  ████████████████████            │   │
│  │  Server 2: 5 连接   ██████████                      │   │
│  │  Server 3: 8 连接   ████████████████                │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  新请求 → 选择 Server 2 (连接数最少)                        │
│                                                             │
│  优势: 动态负载均衡，自动适应服务器处理能力                │
│  适用: 长连接场景 (WebSocket, 数据库连接)                  │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 最少连接策略
# ============================================================
print("=" * 60)
print("最少连接策略")
print("=" * 60)

class LeastConnectionsStrategy(LoadBalanceStrategy):
    """最少连接策略: 选择当前连接数最少的服务器"""
    
    def select(self, servers: List[Server], client_ip: str = None) -> Server:
        healthy = self._get_healthy_servers(servers)
        if not healthy:
            raise RuntimeError("No healthy servers available")
        
        # 选择连接数最少的服务器
        return min(healthy, key=lambda s: s.connections)


# 测试最少连接策略
lc_strategy = LeastConnectionsStrategy()

lc_servers = [
    Server(url="http://server1:8000"),
    Server(url="http://server2:8000"),
    Server(url="http://server3:8000"),
]

# 设置不同的连接数
lc_servers[0].connections = 10
lc_servers[1].connections = 5
lc_servers[2].connections = 8

print("\n服务器连接数:")
for s in lc_servers:
    print(f"  {s.url}: {s.connections} 连接")

selected = lc_strategy.select(lc_servers)
print(f"\n选择: {selected.url} (连接数最少)")

<cell_type>markdown</cell_type>### 2.4 IP 哈希策略 (IP Hash)

**核心概念**: 相同 IP 的请求总是路由到相同服务器，实现会话保持

```
┌─────────────────────────────────────────────────────────────┐
│                    IP 哈希策略                               │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  原理:                                                      │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  hash(client_ip) % server_count = server_index      │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  示例:                                                      │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  192.168.1.100 → hash → Server 1                    │   │
│  │  192.168.1.101 → hash → Server 2                    │   │
│  │  10.0.0.1      → hash → Server 3                    │   │
│  │                                                     │   │
│  │  同一 IP 的所有请求总是路由到同一服务器             │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  优势: 会话保持，无需共享 Session                          │
│  适用: 有状态服务，需要会话亲和性                          │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# IP 哈希策略
# ============================================================
print("=" * 60)
print("IP 哈希策略")
print("=" * 60)

class IPHashStrategy(LoadBalanceStrategy):
    """IP 哈希策略: 相同 IP 的请求总是路由到相同服务器"""
    
    def select(self, servers: List[Server], client_ip: str = None) -> Server:
        healthy = self._get_healthy_servers(servers)
        if not healthy:
            raise RuntimeError("No healthy servers available")
        
        if not client_ip:
            return random.choice(healthy)
        
        # 计算 IP 哈希
        hash_value = int(hashlib.md5(client_ip.encode()).hexdigest(), 16)
        return healthy[hash_value % len(healthy)]


# 测试 IP 哈希策略
ip_strategy = IPHashStrategy()

ip_servers = [
    Server(url="http://server1:8000"),
    Server(url="http://server2:8000"),
    Server(url="http://server3:8000"),
]

# 测试相同 IP 的一致性
test_ips = ["192.168.1.100", "192.168.1.101", "10.0.0.1"]

print("\nIP 哈希一致性测试:")
for ip in test_ips:
    results = set()
    for _ in range(10):
        selected = ip_strategy.select(ip_servers, client_ip=ip)
        results.add(selected.url)
    print(f"  IP {ip}: 总是路由到 {list(results)[0]}")

<cell_type>markdown</cell_type>### 2.5 响应时间策略

**核心概念**: 选择平均响应时间最短的服务器，适用于延迟敏感场景

```
┌─────────────────────────────────────────────────────────────┐
│                    响应时间策略                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  服务器响应时间:                                            │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  Server 1: 100ms  ████████████████████████████████  │   │
│  │  Server 2: 20ms   ██████                            │   │
│  │  Server 3: 50ms   ████████████████                  │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  新请求 → 选择 Server 2 (响应最快)                          │
│                                                             │
│  优势: 自动选择性能最好的服务器                            │
│  适用: 延迟敏感场景，服务器性能差异大                      │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 响应时间策略
# ============================================================
print("=" * 60)
print("响应时间策略")
print("=" * 60)

class ResponseTimeStrategy(LoadBalanceStrategy):
    """响应时间策略: 选择平均响应时间最短的服务器"""
    
    def select(self, servers: List[Server], client_ip: str = None) -> Server:
        healthy = self._get_healthy_servers(servers)
        if not healthy:
            raise RuntimeError("No healthy servers available")
        
        # 选择响应时间最短的服务器
        return min(healthy, key=lambda s: s.avg_response_time if s.avg_response_time > 0 else float('inf'))


# 测试响应时间策略
rt_strategy = ResponseTimeStrategy()

rt_servers = [
    Server(url="http://server1:8000"),
    Server(url="http://server2:8000"),
    Server(url="http://server3:8000"),
]

# 模拟不同的响应时间
for _ in range(10):
    rt_servers[0].record_response(100.0, True)  # 慢
    rt_servers[1].record_response(20.0, True)   # 快
    rt_servers[2].record_response(50.0, True)   # 中等

print("\n服务器响应时间:")
for s in rt_servers:
    print(f"  {s.url}: {s.avg_response_time:.1f}ms")

selected = rt_strategy.select(rt_servers)
print(f"\n选择: {selected.url} (响应最快)")

<cell_type>markdown</cell_type>## 3. 限流器 (Rate Limiter)

**核心概念**: 使用令牌桶算法实现限流，防止服务过载

```
┌─────────────────────────────────────────────────────────────┐
│                    令牌桶算法                                │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  原理:                                                      │
│  ┌─────────────────────────────────────────────────────┐   │
│  │                                                     │   │
│  │     令牌以固定速率生成                              │   │
│  │            ↓                                        │   │
│  │     ┌─────────────────┐                            │   │
│  │     │  ○ ○ ○ ○ ○ ○    │  ← 桶容量限制最大令牌数   │   │
│  │     │  ○ ○ ○ ○        │                            │   │
│  │     │  ○ ○            │                            │   │
│  │     └────────┬────────┘                            │   │
│  │              │                                      │   │
│  │              ▼                                      │   │
│  │         请求消耗令牌                                │   │
│  │         有令牌 → 通过                               │   │
│  │         无令牌 → 拒绝                               │   │
│  │                                                     │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  参数:                                                      │
│  - rate: 令牌生成速率 (令牌/秒)                            │
│  - capacity: 桶容量 (允许突发流量)                         │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 限流器 (自包含实现)
# ============================================================
print("=" * 60)
print("限流器 (令牌桶算法)")
print("=" * 60)

class RateLimiter:
    """
    令牌桶限流器
    
    原理:
    - 令牌以固定速率生成
    - 桶有最大容量限制
    - 每个请求消耗一个令牌
    - 无令牌时请求被拒绝
    """
    
    def __init__(self, rate: float, capacity: int):
        """
        参数:
            rate: 令牌生成速率 (令牌/秒)
            capacity: 桶容量 (最大令牌数)
        """
        self.rate = rate
        self.capacity = capacity
        self.tokens = capacity
        self.last_update = time.time()
    
    async def acquire(self) -> bool:
        """
        尝试获取一个令牌
        
        返回:
            True: 获取成功
            False: 获取失败 (被限流)
        """
        now = time.time()
        elapsed = now - self.last_update
        
        # 补充令牌
        self.tokens = min(self.capacity, self.tokens + elapsed * self.rate)
        self.last_update = now
        
        if self.tokens >= 1:
            self.tokens -= 1
            return True
        return False


# 创建限流器: 每秒 10 个令牌，桶容量 5
limiter = RateLimiter(rate=10.0, capacity=5)

print("\n限流器配置:")
print(f"  速率: {limiter.rate} 令牌/秒")
print(f"  容量: {limiter.capacity} 令牌")

In [ ]:
# ============================================================
# 测试限流器
# ============================================================
print("=" * 60)
print("限流器测试")
print("=" * 60)

async def test_rate_limiter():
    """测试限流器行为"""
    limiter = RateLimiter(rate=10.0, capacity=5)
    
    print("\n快速请求测试 (应该有些被拒绝):")
    results = []
    for i in range(10):
        success = await limiter.acquire()
        results.append(success)
        print(f"  请求 {i+1}: {'✓ 通过' if success else '✗ 拒绝'}")
    
    print(f"\n统计: 通过={sum(results)}, 拒绝={len(results) - sum(results)}")
    
    # 等待令牌补充
    print("\n等待 0.5 秒后再次请求...")
    await asyncio.sleep(0.5)
    
    success = await limiter.acquire()
    print(f"请求: {'✓ 通过' if success else '✗ 拒绝'}")

await test_rate_limiter()

<cell_type>markdown</cell_type>## 4. 熔断器 (Circuit Breaker)

**核心概念**: 熔断器用于故障隔离，防止级联故障

```
┌─────────────────────────────────────────────────────────────┐
│                    熔断器状态机                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  状态转换:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │                                                     │   │
│  │   ┌────────────┐    失败达到阈值    ┌────────────┐ │   │
│  │   │   CLOSED   │ ─────────────────→ │    OPEN    │ │   │
│  │   │   (正常)   │                    │   (熔断)   │ │   │
│  │   └────────────┘                    └─────┬──────┘ │   │
│  │         ↑                                 │        │   │
│  │         │                            超时后        │   │
│  │         │                                 ↓        │   │
│  │         │    成功恢复    ┌────────────────┐        │   │
│  │         └───────────────│   HALF_OPEN    │        │   │
│  │                         │    (半开)      │        │   │
│  │                         └────────┬───────┘        │   │
│  │                                  │                │   │
│  │                             失败则重新 OPEN       │   │
│  │                                  ↓                │   │
│  │                         返回 OPEN 状态            │   │
│  │                                                     │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  参数:                                                      │
│  - failure_threshold: 触发熔断的连续失败次数               │
│  - recovery_timeout: 熔断后等待恢复的时间                  │
│  - half_open_requests: 半开状态允许的测试请求数            │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 熔断器 (自包含实现)
# ============================================================
print("=" * 60)
print("熔断器")
print("=" * 60)

class CircuitBreakerState(Enum):
    """熔断器状态"""
    CLOSED = "closed"      # 正常状态
    OPEN = "open"          # 熔断状态
    HALF_OPEN = "half_open"  # 半开状态


class CircuitBreaker:
    """
    熔断器
    
    用于故障隔离，防止级联故障
    
    状态转换:
    - CLOSED → OPEN: 连续失败达到阈值
    - OPEN → HALF_OPEN: 超时后尝试恢复
    - HALF_OPEN → CLOSED: 成功恢复
    - HALF_OPEN → OPEN: 恢复失败
    """
    
    def __init__(
        self,
        failure_threshold: int = 5,
        recovery_timeout: float = 30.0,
        half_open_requests: int = 3
    ):
        """
        参数:
            failure_threshold: 触发熔断的连续失败次数
            recovery_timeout: 熔断后等待恢复的时间 (秒)
            half_open_requests: 半开状态允许的测试请求数
        """
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.half_open_requests = half_open_requests
        
        self.state = CircuitBreakerState.CLOSED
        self.failure_count = 0
        self.success_count = 0
        self.last_failure_time = 0.0
    
    async def can_execute(self) -> bool:
        """检查是否可以执行请求"""
        if self.state == CircuitBreakerState.CLOSED:
            return True
        
        if self.state == CircuitBreakerState.OPEN:
            # 检查是否超时，可以尝试恢复
            if time.time() - self.last_failure_time > self.recovery_timeout:
                self.state = CircuitBreakerState.HALF_OPEN
                self.success_count = 0
                return True
            return False
        
        # HALF_OPEN 状态
        return True
    
    async def record_success(self):
        """记录成功"""
        if self.state == CircuitBreakerState.HALF_OPEN:
            self.success_count += 1
            if self.success_count >= self.half_open_requests:
                self.state = CircuitBreakerState.CLOSED
                self.failure_count = 0
        else:
            self.failure_count = 0
    
    async def record_failure(self):
        """记录失败"""
        self.failure_count += 1
        self.last_failure_time = time.time()
        
        if self.state == CircuitBreakerState.HALF_OPEN:
            self.state = CircuitBreakerState.OPEN
        elif self.failure_count >= self.failure_threshold:
            self.state = CircuitBreakerState.OPEN


# 创建熔断器
breaker = CircuitBreaker(
    failure_threshold=3,      # 3 次失败触发熔断
    recovery_timeout=1.0,     # 1 秒后尝试恢复
    half_open_requests=2      # 半开状态允许 2 个请求
)

print("\n熔断器配置:")
print(f"  失败阈值: {breaker.failure_threshold}")
print(f"  恢复超时: {breaker.recovery_timeout}s")
print(f"  半开请求数: {breaker.half_open_requests}")
print(f"  当前状态: {breaker.state.value}")

In [ ]:
# ============================================================
# 测试熔断器状态转换
# ============================================================
print("=" * 60)
print("熔断器状态转换测试")
print("=" * 60)

async def test_circuit_breaker():
    """测试熔断器状态转换"""
    breaker = CircuitBreaker(
        failure_threshold=3,
        recovery_timeout=0.5,
        half_open_requests=2
    )
    
    print("\n1. 初始状态 (CLOSED):")
    print(f"   状态: {breaker.state.value}")
    print(f"   可执行: {await breaker.can_execute()}")
    
    print("\n2. 记录 3 次失败:")
    for i in range(3):
        await breaker.record_failure()
        print(f"   失败 {i+1}: 状态={breaker.state.value}")
    
    print("\n3. 熔断状态 (OPEN):")
    print(f"   状态: {breaker.state.value}")
    print(f"   可执行: {await breaker.can_execute()}")
    
    print("\n4. 等待恢复超时...")
    await asyncio.sleep(0.6)
    
    print("\n5. 半开状态 (HALF_OPEN):")
    can_exec = await breaker.can_execute()
    print(f"   状态: {breaker.state.value}")
    print(f"   可执行: {can_exec}")
    
    print("\n6. 记录成功，恢复正常:")
    await breaker.record_success()
    await breaker.record_success()
    print(f"   状态: {breaker.state.value}")

await test_circuit_breaker()

<cell_type>markdown</cell_type>## 5. 完整负载均衡器

**核心概念**: LoadBalancer 类整合了所有功能：负载均衡策略、健康检查、限流、熔断

```
┌─────────────────────────────────────────────────────────────┐
│                    完整负载均衡器架构                        │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  请求 → [限流器] → [熔断器] → [策略选择] → [服务器]        │
│                                                             │
│  组件:                                                      │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  1. 服务器列表: 后端服务器节点                      │   │
│  │  2. 负载均衡策略: 轮询/加权/最少连接等              │   │
│  │  3. 限流器: 防止过载                                │   │
│  │  4. 熔断器: 故障隔离                                │   │
│  │  5. 健康检查: 自动检测服务器状态                    │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  请求处理流程:                                              │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  1. 限流检查: 是否超过速率限制                      │   │
│  │  2. 熔断检查: 是否处于熔断状态                      │   │
│  │  3. 选择服务器: 根据策略选择健康服务器              │   │
│  │  4. 发送请求: 转发请求到选中的服务器                │   │
│  │  5. 记录结果: 更新统计信息                          │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 完整负载均衡器 (自包含实现)
# ============================================================
print("=" * 60)
print("完整负载均衡器")
print("=" * 60)

class LoadBalanceStrategyType(Enum):
    """负载均衡策略类型"""
    ROUND_ROBIN = "round_robin"
    WEIGHTED_ROUND_ROBIN = "weighted_round_robin"
    LEAST_CONNECTIONS = "least_connections"
    IP_HASH = "ip_hash"
    RESPONSE_TIME = "response_time"
    RANDOM = "random"


class LoadBalancer:
    """
    完整负载均衡器
    
    整合负载均衡策略、健康检查、限流、熔断
    """
    
    def __init__(
        self,
        servers: List[str],
        strategy: str = "round_robin",
        weights: List[int] = None,
        enable_circuit_breaker: bool = True,
        rate_limit: float = None
    ):
        """
        参数:
            servers: 服务器 URL 列表
            strategy: 负载均衡策略
            weights: 服务器权重列表
            enable_circuit_breaker: 是否启用熔断器
            rate_limit: 限流速率 (请求/秒)
        """
        # 创建服务器列表
        self.servers = []
        for i, url in enumerate(servers):
            weight = weights[i] if weights and i < len(weights) else 1
            self.servers.append(Server(url=url, weight=weight))
        
        # 设置策略
        self.strategy_type = LoadBalanceStrategyType(strategy)
        self.strategy = self._create_strategy(strategy)
        
        # 熔断器
        self.enable_circuit_breaker = enable_circuit_breaker
        self.circuit_breaker = CircuitBreaker() if enable_circuit_breaker else None
        
        # 限流器
        self.rate_limiter = RateLimiter(rate=rate_limit, capacity=int(rate_limit)) if rate_limit else None
    
    def _create_strategy(self, strategy: str) -> LoadBalanceStrategy:
        """创建负载均衡策略"""
        strategies = {
            "round_robin": RoundRobinStrategy,
            "weighted_round_robin": WeightedRoundRobinStrategy,
            "least_connections": LeastConnectionsStrategy,
            "ip_hash": IPHashStrategy,
            "response_time": ResponseTimeStrategy,
        }
        return strategies.get(strategy, RoundRobinStrategy)()
    
    def select_server(self, client_ip: str = None) -> Server:
        """选择服务器"""
        return self.strategy.select(self.servers, client_ip)
    
    def get_stats(self) -> Dict[str, Any]:
        """获取统计信息"""
        return {
            "strategy": self.strategy_type.value,
            "servers": [
                {
                    "url": s.url,
                    "healthy": s.healthy,
                    "weight": s.weight,
                    "connections": s.connections,
                    "avg_response_time": s.avg_response_time,
                    "success_rate": s.success_rate,
                }
                for s in self.servers
            ]
        }


# 创建负载均衡器
balancer = LoadBalancer(
    servers=[
        "http://server1:8000",
        "http://server2:8000",
        "http://server3:8000",
    ],
    strategy="round_robin",
    weights=[3, 2, 1],
    enable_circuit_breaker=True,
    rate_limit=100.0,
)

print("\n负载均衡器配置:")
print(f"  策略: {balancer.strategy_type.value}")
print(f"  服务器数: {len(balancer.servers)}")
print(f"  熔断器: {balancer.enable_circuit_breaker}")
print(f"  限流: {balancer.rate_limiter is not None}")

In [ ]:
# ============================================================
# 获取负载均衡器统计信息
# ============================================================
print("=" * 60)
print("负载均衡器统计信息")
print("=" * 60)

# 模拟一些请求来生成统计数据
for i in range(30):
    server = balancer.select_server(client_ip=f"192.168.1.{i % 10}")
    server.connections += 1
    # 模拟响应
    latency = np.random.exponential(10)
    success = np.random.random() > 0.05
    server.record_response(latency, success)
    server.connections -= 1

# 获取统计信息
stats = balancer.get_stats()

print(f"\n策略: {stats['strategy']}")
print(f"\n服务器状态:")
print("-" * 60)
for server in stats['servers']:
    print(f"\n  {server['url']}:")
    print(f"    健康状态: {'✓' if server['healthy'] else '✗'}")
    print(f"    权重: {server['weight']}")
    print(f"    当前连接: {server['connections']}")
    print(f"    平均响应时间: {server['avg_response_time']:.2f}ms")
    print(f"    成功率: {server['success_rate']:.1%}")

<cell_type>markdown</cell_type>## 6. 策略对比

**核心概念**: 根据业务场景选择合适的负载均衡策略

```
┌─────────────────────────────────────────────────────────────┐
│                    策略选择决策树                            │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│                        开始                                 │
│                          │                                  │
│                          ▼                                  │
│                 ┌────────────────┐                          │
│                 │  需要会话保持？ │                          │
│                 └────────┬───────┘                          │
│                          │                                  │
│              ┌───────────┴───────────┐                      │
│              │                       │                      │
│              ▼                       ▼                      │
│             是                      否                      │
│              │                       │                      │
│              ▼                       ▼                      │
│          IP 哈希            ┌────────────────┐              │
│                             │ 服务器性能相近？│              │
│                             └────────┬───────┘              │
│                                      │                      │
│                          ┌───────────┴───────────┐          │
│                          │                       │          │
│                          ▼                       ▼          │
│                         是                      否          │
│                          │                       │          │
│                          ▼                       ▼          │
│                        轮询                  加权轮询       │
│                                                             │
│  其他场景:                                                  │
│  - 长连接场景 → 最少连接                                   │
│  - 延迟敏感 → 响应时间策略                                 │
│  - 无状态服务 → 随机                                       │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 策略对比表
# ============================================================
print("=" * 60)
print("负载均衡策略对比")
print("=" * 60)

print(f"\n{'策略':<15} {'原理':<20} {'优点':<20} {'适用场景'}")
print("-" * 75)

strategies = [
    ("轮询", "依次选择", "简单公平", "服务器性能相近"),
    ("加权轮询", "按权重分配", "考虑服务器能力", "服务器性能不同"),
    ("最少连接", "选择连接最少", "动态负载均衡", "长连接场景"),
    ("IP 哈希", "IP 哈希取模", "会话保持", "有状态服务"),
    ("响应时间", "选择响应最快", "性能优先", "延迟敏感场景"),
    ("随机", "随机选择", "简单高效", "无状态服务"),
]

for name, principle, pros, scenario in strategies:
    print(f"{name:<15} {principle:<20} {pros:<20} {scenario}")

print("\n策略选择建议:")
print("  1. 默认使用轮询策略，简单可靠")
print("  2. 服务器配置不同时使用加权轮询")
print("  3. 需要会话保持时使用 IP 哈希")
print("  4. 长连接场景使用最少连接")
print("  5. 延迟敏感场景使用响应时间策略")

<cell_type>markdown</cell_type>## 7. 生产部署示例

**核心概念**: 生产环境通常使用 NGINX 或 Kubernetes 实现负载均衡

```
┌─────────────────────────────────────────────────────────────┐
│                    生产部署架构                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  方案 1: NGINX 负载均衡                                     │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  客户端 → NGINX (80/443) → 后端服务器集群           │   │
│  │                                                     │   │
│  │  优点: 高性能、配置灵活、支持 SSL 终止             │   │
│  │  适用: 中小规模部署                                 │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  方案 2: Kubernetes Service                                 │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  客户端 → K8s Service → Pod 集群                   │   │
│  │                                                     │   │
│  │  优点: 自动扩缩容、服务发现、滚动更新              │   │
│  │  适用: 大规模容器化部署                             │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  方案 3: 云负载均衡器                                       │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  客户端 → ALB/NLB/CLB → 后端实例                   │   │
│  │                                                     │   │
│  │  优点: 托管服务、高可用、自动健康检查              │   │
│  │  适用: 云原生部署                                   │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# NGINX 负载均衡配置示例
# ============================================================
print("=" * 60)
print("NGINX 负载均衡配置")
print("=" * 60)

print("""
# nginx.conf - 模型服务负载均衡配置

# 定义后端服务器组
upstream model_servers {
    # 负载均衡策略 (可选: least_conn, ip_hash)
    least_conn;
    
    # 后端服务器列表 (weight 指定权重)
    server model-server-1:8000 weight=3;
    server model-server-2:8000 weight=2;
    server model-server-3:8000 weight=1;
    
    # 保持连接数 (提高性能)
    keepalive 32;
}

server {
    listen 80;
    server_name api.example.com;

    # 推理接口
    location /predict {
        proxy_pass http://model_servers;
        proxy_http_version 1.1;
        proxy_set_header Connection "";
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        
        # 超时配置
        proxy_connect_timeout 10s;
        proxy_read_timeout 60s;
        proxy_send_timeout 60s;
    }

    # 健康检查端点
    location /health {
        return 200 'OK';
        add_header Content-Type text/plain;
    }
    
    # 指标端点
    location /metrics {
        proxy_pass http://model_servers;
    }
}
""")

print("配置说明:")
print("  - least_conn: 最少连接策略")
print("  - weight: 服务器权重")
print("  - keepalive: 保持连接数")
print("  - proxy_read_timeout: 推理超时时间")

In [ ]:
# ============================================================
# Kubernetes Service 配置示例
# ============================================================
print("=" * 60)
print("Kubernetes Service 配置")
print("=" * 60)

print("""
# service.yaml - Kubernetes 负载均衡配置

apiVersion: v1
kind: Service
metadata:
  name: model-server
  labels:
    app: model-server
spec:
  # 服务类型: ClusterIP, NodePort, LoadBalancer
  type: LoadBalancer
  
  # 选择器: 匹配 Pod 标签
  selector:
    app: model-server
  
  # 端口映射
  ports:
  - name: http
    port: 80
    targetPort: 8000
    protocol: TCP
  
  # 会话亲和性 (IP 哈希)
  sessionAffinity: ClientIP
  sessionAffinityConfig:
    clientIP:
      timeoutSeconds: 3600

---
# deployment.yaml - 后端 Pod 部署

apiVersion: apps/v1
kind: Deployment
metadata:
  name: model-server
spec:
  replicas: 3  # 副本数
  selector:
    matchLabels:
      app: model-server
  template:
    metadata:
      labels:
        app: model-server
    spec:
      containers:
      - name: model-server
        image: model-server:latest
        ports:
        - containerPort: 8000
        resources:
          requests:
            memory: "2Gi"
            cpu: "1"
          limits:
            memory: "4Gi"
            cpu: "2"
        # 健康检查
        livenessProbe:
          httpGet:
            path: /health
            port: 8000
          initialDelaySeconds: 30
          periodSeconds: 10
        readinessProbe:
          httpGet:
            path: /ready
            port: 8000
          initialDelaySeconds: 5
          periodSeconds: 5
""")

print("配置说明:")
print("  - type: LoadBalancer 创建云负载均衡器")
print("  - sessionAffinity: ClientIP 实现会话保持")
print("  - replicas: 副本数，可配合 HPA 自动扩缩")
print("  - livenessProbe: 存活检查，失败则重启")
print("  - readinessProbe: 就绪检查，失败则不接收流量")

<cell_type>markdown</cell_type>## 总结

本教程介绍了负载均衡的核心概念和实现：

### 核心知识点

| 主题 | 关键内容 |
|:-----|:---------|
| 服务器节点 | Server 类、健康状态、响应时间统计 |
| 负载均衡策略 | 轮询、加权轮询、最少连接、IP哈希、响应时间 |
| 限流器 | 令牌桶算法、速率控制、突发流量处理 |
| 熔断器 | 状态机 (CLOSED→OPEN→HALF_OPEN)、故障隔离 |
| 生产部署 | NGINX 配置、Kubernetes Service |

### API 速查

```python
# 服务器节点
server = Server(url="http://server:8000", weight=3)
server.record_response(latency_ms, success=True)
print(server.avg_response_time, server.success_rate)

# 负载均衡策略
strategy = RoundRobinStrategy()           # 轮询
strategy = WeightedRoundRobinStrategy()   # 加权轮询
strategy = LeastConnectionsStrategy()     # 最少连接
strategy = IPHashStrategy()               # IP 哈希
strategy = ResponseTimeStrategy()         # 响应时间
selected = strategy.select(servers, client_ip="192.168.1.1")

# 限流器 (令牌桶)
limiter = RateLimiter(rate=100.0, capacity=10)
if await limiter.acquire():
    # 处理请求
    pass

# 熔断器
breaker = CircuitBreaker(failure_threshold=5, recovery_timeout=30.0)
if await breaker.can_execute():
    try:
        result = do_request()
        await breaker.record_success()
    except Exception:
        await breaker.record_failure()

# 完整负载均衡器
balancer = LoadBalancer(
    servers=["http://s1:8000", "http://s2:8000"],
    strategy="round_robin",
    weights=[3, 2],
    enable_circuit_breaker=True,
    rate_limit=100.0
)
server = balancer.select_server(client_ip="192.168.1.1")
stats = balancer.get_stats()
```

### 生产环境检查清单

```
部署前检查:
✓ 选择合适的负载均衡策略
✓ 配置健康检查自动剔除故障节点
✓ 启用熔断器防止级联故障
✓ 配置限流保护后端服务
✓ 设置合理的超时时间
✓ 监控负载均衡器指标

性能指标:
✓ 健康节点比例 > 80%
✓ 请求成功率 > 99%
✓ 负载分布均匀 (标准差 < 10%)
✓ 熔断触发次数监控
```

### 下一步学习

- **04_Advanced_Serving_tutorial.ipynb**: 高级服务技术 (微服务、服务网格、事件驱动)